# Tahap A — Training YOLOv8 Deteksi Tangan (BISINDO)

Notebook ini **hanya** untuk melatih + mengevaluasi model **YOLOv8n** deteksi tangan
(**single-class `Hand`**) di **Google Colab (GPU)**. Model dipakai untuk **ROI cropping**
(deteksi tangan) SEBELUM MediaPipe pada pipeline real-time (itu **Tahap B**, terpisah).

**Output:**
- `best.pt` — bobot YOLO terlatih (dibawa ke project sebagai `models/yolo_hand.pt`).
- Metrik **mAP@50, mAP@50-95, precision, recall** pada **test set** + artefak visual
  (kurva PR, kurva training, confusion matrix, contoh deteksi) untuk lampiran laporan TA.

> Catatan: TIDAK ada webcam / inference real-time di sini. Murni training + evaluasi.
> Jalankan sel **berurutan dari atas ke bawah**.

## Langkah 0 — Pastikan runtime GPU
Menu Colab: **Runtime → Change runtime type → Hardware accelerator → GPU (T4)**.
Sel berikut akan gagal (assert) bila GPU belum aktif.

In [ ]:
# Cek GPU Colab
!nvidia-smi
import torch
assert torch.cuda.is_available(), \
    "GPU tidak aktif! Runtime > Change runtime type > GPU, lalu jalankan ulang."
print("CUDA device :", torch.cuda.get_device_name(0))
print("torch        :", torch.__version__)

In [ ]:
# Install dependency. ultralytics DIPIN agar sama dengan integrasi Tahap B (reproducible).
%pip install -q ultralytics==8.3.0 roboflow
import ultralytics
ultralytics.checks()   # cetak versi + status GPU/RAM

## Konfigurasi
Semua parameter terkumpul di SATU sel di bawah agar mudah diubah. `SEED` tetap untuk
reproducibility. Ganti placeholder Roboflow dengan milikmu.

In [ ]:
import random, numpy as np, torch
from pathlib import Path

# ---- Reproducibility ----
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ---- Hyperparameter training (UBAH DI SINI SAJA) ----
MODEL    = "yolov8n.pt"   # nano: paling ringan, target real-time RTX 2050. Jangan ganti varian besar tanpa alasan.
IMGSZ    = 640
EPOCHS   = 100
PATIENCE = 20             # early stopping: berhenti bila metrik val tak membaik 20 epoch
BATCH    = 16             # aman utk T4 16GB @ yolov8n 640. -1 = auto-batch (bisa kurang stabil)
PROJECT  = "runs_yolo_hand"
NAME     = "yolov8n_hand"

# ---- Roboflow (ISI punyamu) ----
RF_API_KEY   = ""         # <-- API key Roboflow
RF_WORKSPACE = ""         # <-- slug workspace
RF_PROJECT   = ""         # <-- slug project
RF_VERSION   = 1          # <-- nomor versi dataset

print("Config siap | MODEL=%s IMGSZ=%d EPOCHS=%d BATCH=%d SEED=%d"
      % (MODEL, IMGSZ, EPOCHS, BATCH, SEED))

## Dataset — Roboflow (format YOLOv8)

Isi `RF_*` di sel config. Dataset **wajib single-class `Hand`** dan **sudah terbagi
train/valid/test**.

**Bila dataset punya banyak label gestur** (mis. A, B, C, ...): remap jadi satu kelas
di Roboflow → **Health Check / Modify Classes → merge semua kelas ke `Hand`**, lalu
**Generate** versi baru dan **Export (YOLOv8)**. Tujuan TA = deteksi *ada tangan*, bukan
mengenali gestur (gestur = tugas LSTM di pipeline).

**Tanpa SDK** (alternatif): unggah ZIP export YOLOv8 ke Colab atau mount Drive, lalu set
`DATA_YAML = "/path/ke/data.yaml"` manual dan lewati sel download.

In [ ]:
# Download dataset format YOLOv8 dari Roboflow.
from roboflow import Roboflow

rf = Roboflow(api_key=RF_API_KEY)
project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
dataset = project.version(RF_VERSION).download("yolov8")

DATA_DIR  = Path(dataset.location)
DATA_YAML = str(DATA_DIR / "data.yaml")
print("Dataset di :", DATA_DIR)
print("DATA_YAML  :", DATA_YAML)

In [ ]:
# Inspeksi data.yaml — pastikan single-class 'Hand'.
import yaml
with open(DATA_YAML) as f:
    dy = yaml.safe_load(f)
print("nc    :", dy.get("nc"))
print("names :", dy.get("names"))

if dy.get("nc") != 1:
    print("\n[WARNING] Dataset BUKAN single-class (nc != 1).")
    print("TA ini butuh SATU kelas 'Hand'. Remap di Roboflow (merge semua kelas ke")
    print("'Hand'), generate versi baru, export ulang YOLOv8, lalu ulangi sel download.")
else:
    print("\nSingle-class OK ->", dy.get("names"))

## Verifikasi split + cek leakage

**Leakage** (gambar/frame yang sama muncul di >1 split) membuat **mAP tidak valid** —
model seolah hebat padahal sudah "melihat" data test saat training. Sel berikut:
1. menghitung jumlah gambar per split, dan
2. mendeteksi **duplikat IDENTIK lintas-split** via md5 (bukti anti-leakage).

In [ ]:
import hashlib, collections

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def img_files(split):
    d = DATA_DIR / split / "images"
    return [p for p in d.glob("*") if p.suffix.lower() in IMG_EXT] if d.exists() else []

splits = {s: img_files(s) for s in ["train", "valid", "test"]}
print(f"{'split':<8} jumlah_gambar")
for s, fs in splits.items():
    print(f"{s:<8} {len(fs)}")
if not splits["test"]:
    print("\n[WARNING] split 'test' kosong/absen — evaluasi WAJIB pakai test set!")

# --- duplikat EXACT lintas-split (md5) ---
def md5(p): return hashlib.md5(p.read_bytes()).hexdigest()

h2split = collections.defaultdict(set)
h2files = collections.defaultdict(list)
for s, fs in splits.items():
    for p in fs:
        h = md5(p); h2split[h].add(s); h2files[h].append(str(p))

cross = {h: h2files[h] for h, ss in h2split.items() if len(ss) > 1}
print("\n=== CEK LEAKAGE (gambar identik lintas-split) ===")
if not cross:
    print("PASS: tidak ada gambar identik lintas-split.")
else:
    print(f"FAIL: {len(cross)} gambar muncul di >1 split (LEAKAGE -> mAP TAK VALID):")
    for h, files in list(cross.items())[:20]:
        for fp in files: print("   ", fp)

print("\nCATATAN: md5 hanya menangkap duplikat IDENTIK. Frame near-duplikat dari video")
print("yang sama (beda sedikit) TAK terdeteksi — pastikan split dibuat di level sumber/video.")

## Training YOLOv8n

`plots=True` membuat Ultralytics otomatis menyimpan kurva training, PR curve, dan
confusion matrix di run directory. Early stopping (`patience`) menghentikan training
bila metrik validasi tak membaik.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    seed=SEED,
    project=PROJECT,
    name=NAME,
    plots=True,
)
RUN_DIR = Path(results.save_dir)
BEST_PT = RUN_DIR / "weights" / "best.pt"
print("\nRun dir :", RUN_DIR)
print("best.pt :", BEST_PT, "| ada:", BEST_PT.exists())

## Evaluasi pada TEST set

Evaluasi memakai **split `test`** (BUKAN val) — inilah angka yang dilaporkan di TA.
Mengambil mAP@50, mAP@50-95, precision, recall.

In [ ]:
import json

best = YOLO(str(BEST_PT))
metrics = best.val(
    data=DATA_YAML, split="test", imgsz=IMGSZ,
    project=PROJECT, name=NAME + "_test", plots=True,
)

res = {
    "mAP@50":    float(metrics.box.map50),
    "mAP@50-95": float(metrics.box.map),
    "precision": float(metrics.box.mp),
    "recall":    float(metrics.box.mr),
}
print("=== METRIK TEST SET ===")
for k, v in res.items():
    print(f"{k:<11}: {v:.4f}")

metrics_path = RUN_DIR / "metrics_test.json"
metrics_path.write_text(json.dumps(res, indent=2))
print("\nDisimpan:", metrics_path)

## Artefak training (untuk lampiran TA)
Kurva training, PR curve, confusion matrix, dan kurva P/R/F1 — auto-generate oleh Ultralytics.

In [ ]:
from IPython.display import Image, display

arts = ["results.png", "PR_curve.png", "confusion_matrix.png",
        "P_curve.png", "R_curve.png", "F1_curve.png"]
for a in arts:
    fp = RUN_DIR / a
    if fp.exists():
        print("•", a); display(Image(filename=str(fp)))
    else:
        print("(tidak ada)", a)

## Contoh deteksi kualitatif
Bukti visual model benar-benar mendeteksi tangan (bukan menghafal angka mAP). Bounding
box digambar pada beberapa gambar test.

In [ ]:
import random as _r

test_imgs = splits["test"] or splits["valid"]
sample = _r.sample(test_imgs, min(6, len(test_imgs)))
pred = best.predict(
    [str(p) for p in sample], imgsz=IMGSZ, conf=0.25,
    save=True, project=PROJECT, name=NAME + "_pred",
)
pred_dir = Path(pred[0].save_dir)
print("Hasil deteksi disimpan:", pred_dir)
for p in sorted(pred_dir.glob("*")):
    if p.suffix.lower() in IMG_EXT:
        display(Image(filename=str(p)))

## PENTING — interpretasi mAP tinggi (bekal sidang)

Jika **mAP@50 sangat tinggi (mis. > 0.98)**, itu **WAJAR** untuk **single-class** dengan
objek dominan (tangan besar & jelas di frame) — bukan otomatis pertanda curang.

Yang harus bisa kamu jelaskan ke penguji:
- **Test set benar-benar terpisah** & tak pernah dilihat saat training — dibuktikan oleh
  sel cek leakage (duplikat identik lintas-split = 0).
- **Bukti kualitatif** (contoh deteksi di atas) menunjukkan box pas di tangan pada gambar
  yang belum pernah dilatih → model bekerja, bukan menghafal.
- Split idealnya dibuat **di level sumber/video** agar frame-frame dari satu rekaman tak
  tersebar ke train & test sekaligus (near-duplicate leakage).

In [ ]:
from google.colab import drive, files
import shutil

# 1) Salin best.pt + metrik ke Google Drive (nama target = yang dipakai Tahap B)
drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive/bisindo_yolo")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
drive_best = DRIVE_DIR / "yolo_hand.pt"
shutil.copy(BEST_PT, drive_best)
shutil.copy(RUN_DIR / "metrics_test.json", DRIVE_DIR / "metrics_test.json")
print("best.pt -> Drive:", drive_best)

# 2) Download langsung ke laptop
files.download(str(BEST_PT))

# 3) Ringkasan akhir
print("\n==================== RINGKASAN AKHIR ====================")
print("best.pt    :", BEST_PT)
print("Drive      :", drive_best)
print(f"mAP@50     : {res['mAP@50']:.4f}")
print(f"mAP@50-95  : {res['mAP@50-95']:.4f}")
print(f"precision  : {res['precision']:.4f}")
print(f"recall     : {res['recall']:.4f}")
print("Plots/run  :", RUN_DIR)
print("Metrik JSON:", RUN_DIR / "metrics_test.json")
print("========================================================")

## Ringkasan & checklist lampiran TA

**Artefak untuk laporan** (ada di `RUN_DIR`):
- `results.png` (kurva loss + mAP per epoch), `PR_curve.png`, `confusion_matrix.png`,
  `P/R/F1_curve.png`, folder `*_pred` (contoh deteksi), `metrics_test.json`.

**Bawa ke Tahap B (integrasi project):**
1. Ambil `best.pt` (dari download atau Drive `bisindo_yolo/yolo_hand.pt`).
2. Salin ke project sebagai **`models/yolo_hand.pt`**.
3. Set `config.USE_YOLO_ROI = True`, jalankan, lalu verifikasi parity prediksi (ON vs OFF).

Selesai — punya `best.pt`, metrik mAP, dan analisis untuk laporan.